# 5. Résumé automatique d'un département avec un LLM

**Principe : le code calcule, le LLM rédige.**

1. **SQL** calcule les chiffres clés d'un département : ils sont fiables.
2. On les rassemble dans une **fiche de faits** (un simple texte).
3. Le **LLM** transforme cette fiche en un paragraphe lisible.
4. On **vérifie** automatiquement que le LLM n'a pas inventé de chiffres.

In [1]:
import re
import sqlite3

import ollama
import pandas as pd

con = sqlite3.connect("../data/accidents.db")
MODELE = "qwen2.5:3b"


def sql(requete, params=()):
    """Exécute une requête SQL (avec paramètres optionnels) et renvoie un DataFrame."""
    return pd.read_sql(requete, con, params=params)


def fr(x, decimales=0):
    """Formate un nombre à la française : 1627 -> '1 627', 3.55 -> '3,55'."""
    texte = f"{x:,.{decimales}f}"                 # format anglais : '1,627' ou '3.55'
    return texte.replace(",", " ").replace(".", ",")

## Étape 1 : le code calcule

Requêtes de l'analyse, avec un filtre `WHERE a.dep = ?` sur le département.
Le calcul de la part de tués revient partout : on le stocke une fois dans `PCT_TUES` pour ne pas le réécrire.

In [2]:
PCT_TUES = "ROUND(100.0 * SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) / COUNT(*), 2)"


def stats_departement(dep):
    """Calcule les chiffres clés d'un département et les renvoie dans un dictionnaire."""
    stats = {"dep": dep}

    # Chiffres globaux du département
    kpi = sql(f"""
        SELECT COUNT(DISTINCT a.Num_Acc) AS nb_accidents,
               COUNT(*) AS nb_usagers,
               SUM(CASE WHEN u.gravite = 'Tué' THEN 1 ELSE 0 END) AS nb_tues,
               {PCT_TUES} AS pct_tues
        FROM usagers AS u
        JOIN accidents AS a ON u.Num_Acc = a.Num_Acc
        WHERE a.dep = ?
    """, (dep,)).iloc[0]
    stats.update(kpi.to_dict())          # ajoute les 4 colonnes au dictionnaire

    # Moyenne nationale, pour comparer
    stats["pct_tues_france"] = sql(f"SELECT {PCT_TUES} AS pct FROM usagers AS u").iloc[0]["pct"]

    # Heure avec le plus d'accidents
    pic = sql("""
        SELECT heure, COUNT(*) AS nb FROM accidents
        WHERE dep = ? GROUP BY heure ORDER BY nb DESC LIMIT 1
    """, (dep,)).iloc[0]
    stats["heure_pic"] = int(pic["heure"])

    # Part de tués par période et par type de lieu -> dictionnaires {catégorie: pourcentage}
    for colonne in ["periode", "agglomeration"]:
        df = sql(f"""
            SELECT a.{colonne} AS categorie, {PCT_TUES} AS pct
            FROM usagers AS u
            JOIN accidents AS a ON u.Num_Acc = a.Num_Acc
            WHERE a.dep = ?
            GROUP BY a.{colonne}
        """, (dep,))
        stats[colonne] = dict(zip(df["categorie"], df["pct"]))

    return stats


stats_departement("69")

{'dep': '69',
 'nb_accidents': 1627.0,
 'nb_usagers': 3836.0,
 'nb_tues': 52.0,
 'pct_tues': 1.36,
 'pct_tues_france': np.float64(2.74),
 'heure_pic': 18,
 'periode': {'Crépuscule': 0.8, 'Jour': 1.17, 'Nuit': 1.81},
 'agglomeration': {'En agglomération': 0.81, 'Hors agglomération': 3.17}}

## Étape 2 : la fiche de faits

On transforme le dictionnaire en texte. **Tous les calculs ET toutes les comparaisons sont faits ici, par le code.**

*Pourquoi ?* Dans une première version, la fiche donnait seulement les chiffres (« 1,36 % (moyenne en France : 2,74 %) »).
Le modèle 3B a alors commis des erreurs **sans inventer un seul chiffre**, ce que le garde-fou ne pouvait donc pas détecter :
- **sens de comparaison inversé** : « 1,36 %, soit plus que la moyenne de 2,74 % » ;
- **pourcentage mal interprété** : « 3,17 % des personnes tuées l'ont été hors agglomération », alors qu'il s'agit de la part de tués *parmi les personnes impliquées* hors agglomération ;
- **chiffres attribués à la mauvaise donnée** : le rapport 4,9 présenté comme un « taux national » ;
- **causes inventées** : « les conducteurs sont moins attentifs la nuit ».

D'où cette version, où chaque phrase est **complète et sans ambiguïté**, et où le sens des comparaisons (« inférieure à », « 3,9 fois plus ») est **décidé par le code**.

In [3]:
def position(valeur, reference):
    """Compare deux valeurs EN PYTHON et renvoie le mot à utiliser (le LLM n'a plus à comparer)."""
    if valeur > reference * 1.1:
        return "supérieure à"
    if valeur < reference * 0.9:
        return "inférieure à"
    return "proche de"


def rapport(a, b):
    """Formule le rapport entre deux pourcentages : '3,9 fois plus', 'moins' ou 'autant'."""
    if a >= b * 1.1:
        return f"{fr(a / b, 1)} fois plus"
    if a <= b * 0.9:
        return "moins"
    return "autant"


def fiche(stats):
    """Fiche de faits : chaque phrase est complète, sans ambiguïté, et les comparaisons sont déjà faites."""
    pct, pct_fr = stats["pct_tues"], stats["pct_tues_france"]
    lignes = [
        f"Département {stats['dep']}, année 2024.",
        f"{fr(stats['nb_accidents'])} accidents, impliquant {fr(stats['nb_usagers'])} personnes, "
        f"dont {fr(stats['nb_tues'])} ont été tuées.",
        f"Part des personnes impliquées qui ont été tuées : {fr(pct, 2)} %, "
        f"{position(pct, pct_fr)} la moyenne française ({fr(pct_fr, 2)} %).",
        f"L'heure qui compte le plus d'accidents est {stats['heure_pic']} h.",
    ]

    p = stats["periode"]
    if p.get("Nuit") and p.get("Jour"):
        lignes.append(f"Parmi les personnes impliquées dans un accident de nuit, {fr(p['Nuit'], 2)} % ont été tuées, "
                      f"contre {fr(p['Jour'], 2)} % le jour : les accidents de nuit sont "
                      f"{rapport(p['Nuit'], p['Jour'])} meurtriers que ceux de jour.")

    a = stats["agglomeration"]
    hors, en = a.get("Hors agglomération"), a.get("En agglomération")
    if hors and en:
        lignes.append(f"Parmi les personnes impliquées hors agglomération, {fr(hors, 2)} % ont été tuées, "
                      f"contre {fr(en, 2)} % en agglomération : les accidents hors agglomération sont "
                      f"{rapport(hors, en)} meurtriers.")

    return "\n".join("- " + ligne for ligne in lignes)


print(fiche(stats_departement("69")))

- Département 69, année 2024.
- 1 627 accidents, impliquant 3 836 personnes, dont 52 ont été tuées.
- Part des personnes impliquées qui ont été tuées : 1,36 %, inférieure à la moyenne française (2,74 %).
- L'heure qui compte le plus d'accidents est 18 h.
- Parmi les personnes impliquées dans un accident de nuit, 1,81 % ont été tuées, contre 1,17 % le jour : les accidents de nuit sont 1,5 fois plus meurtriers que ceux de jour.
- Parmi les personnes impliquées hors agglomération, 3,17 % ont été tuées, contre 0,81 % en agglomération : les accidents hors agglomération sont 3,9 fois plus meurtriers.


## Étape 3 : le LLM rédige

Le **prompt système** fixe des règles strictes. C'est notre première protection contre les hallucinations.
`temperature: 0` et `seed: 42` rendent la réponse stable : relancer donne le même texte.

In [4]:
SYSTEME = """Tu es analyste en sécurité routière. À partir de la fiche de données fournie,
rédige un paragraphe de 4 à 5 phrases en français, clair et factuel, destiné au grand public.

Règles strictes :
- Utilise UNIQUEMENT les chiffres de la fiche, recopiés exactement. N'invente aucun chiffre et ne fais aucun calcul.
- Désigne le département uniquement par son numéro (exemple : « le département 69 »).
- Ne fais AUCUNE comparaison toi-même : reprends exactement celles de la fiche (« inférieure à », « 3,9 fois plus »…).
- Ne change pas le sens des pourcentages : reprends les formulations de la fiche.
- N'avance aucune explication ou cause qui ne figure pas dans la fiche."""


def resumer(dep, systeme=SYSTEME, modele=MODELE):
    """Calcule la fiche d'un département puis demande au LLM de la rédiger."""
    faits = fiche(stats_departement(dep))
    reponse = ollama.chat(
        model=modele,
        messages=[
            {"role": "system", "content": systeme},
            {"role": "user", "content": "Fiche de données :\n" + faits},
        ],
        options={"temperature": 0, "seed": 42},
    )
    return faits, reponse["message"]["content"]

## Étape 4 : vérifier automatiquement

Un **garde-fou** simple : on repère tous les nombres du texte généré et on vérifie qu'ils figurent bien dans la fiche.
Un nombre absent de la fiche est **probablement inventé**.

`re` est le module des **expressions régulières**, un mini-langage pour rechercher des motifs dans un texte.
Ici, le motif signifie : « des chiffres, éventuellement suivis de groupes de 3 chiffres séparés par une espace (1 627),
éventuellement suivis d'une virgule et de décimales (3,55) ».

In [5]:
def nombres(texte):
    """Renvoie l'ensemble des nombres présents dans un texte, normalisés ('1 627' -> '1627')."""
    trouves = re.findall(r"\d+(?:[ \u202f\u00a0]\d{3})*(?:[.,]\d+)?", texte)
    return {re.sub(r"[ \u202f\u00a0]", "", t).replace(".", ",") for t in trouves}


def verifier(faits, texte):
    """Affiche les nombres du texte qui n'apparaissent pas dans la fiche."""
    inventes = nombres(texte) - nombres(faits)       # différence entre deux ensembles
    if inventes:
        print("⚠️ Nombres absents de la fiche :", sorted(inventes))
    else:
        print("✅ Tous les nombres du texte viennent de la fiche.")

## Test sur trois départements très différents

- **69** (Rhône) : une grande métropole
- **75** (Paris) : 100 % urbain
- **23** (Creuse) : un département rural

In [6]:
for dep in ["69", "75", "23"]:
    faits, texte = resumer(dep)
    print(f"=============== Département {dep}")
    print(texte)
    verifier(faits, texte)
    print()

=============== Département 69
Le département 69 a enregistré 1 627 accidents en 2024, impliquant 3 836 personnes dont 52 ont perdu la vie. Le taux de mortalité est inférieur à la moyenne française (2,74 %), avec une part des victimes qui a été tuées de 1,36 %. Les accidents nocturnes sont 1,5 fois plus meurtriers que ceux en journée, tandis qu'entre les personnes impliquées hors agglomération et celles dans l'agglomération, ceux-ci ont été 3,9 fois plus meurtriers.
✅ Tous les nombres du texte viennent de la fiche.

=============== Département 75
Le département 75 a enregistré 4 191 accidents en 2024, impliquant 9 132 personnes dont 31 ont été tuées. Le taux de personnes tuées est inférieur à la moyenne française (2,74 %), ce qui est une bonne nouvelle pour cette région. Les accidents se produisent principalement aux alentours de 18 heures. Par ailleurs, les accidents nocturnes sont moins meurtriers que ceux matinaux : seulement 0,30 % des personnes impliquées ont été tuées en nuit, co

## Expérience : sans les règles

Même fiche, mais avec un prompt système minimal. Compare le résultat : le modèle respecte-t-il encore les chiffres ?
Ajoute-t-il des explications ou un nom de département de lui-même ?

In [7]:
faits, texte = resumer("23", systeme="Résume ces données.")
print(texte)
verifier(faits, texte)

Voici une résumation des données :

- En 2024 dans le département 23, on a observé 73 accidents impliquant 148 personnes, dont 9 ont perdu la vie.
- Les accidents de nuit sont plus meurtriers que ceux du jour (9,30% vs. 3,45%), et les accidents hors agglomération sont plus meurtriers que ceux en agglomération (8,70% vs. 1,79%).
- Les heures de la journée qui comptent le plus d'accidents sont celles du soir (16h), avec une proportion de personnes tuées supérieure à la moyenne française.
- Le taux de personnes tuées parmi les victimes est supérieur à la moyenne nationale, atteignant 6,08% contre 2,74%.
✅ Tous les nombres du texte viennent de la fiche.


## Expérience : un modèle plus gros

Même fiche, même prompt, avec `qwen2.5-coder:7b` (7 milliards de paramètres au lieu de 3).
Il est spécialisé en code, mais devrait mieux respecter les consignes. Compare la fidélité des textes : un modèle plus gros est **plus lent**, mais commet-il moins d'erreurs ?

In [8]:
for modele in ["qwen2.5:3b", "qwen2.5-coder:7b"]:
    for dep in ["69", "75", "23"]:
        faits, texte = resumer(dep, modele=modele)
        print(f"=============== {modele} · département {dep}")
        print(texte)
        verifier(faits, texte)
        print()

=============== qwen2.5:3b · département 69
Le département 69 a enregistré 1 627 accidents en 2024, impliquant 3 836 personnes dont 52 ont perdu la vie. Le taux de mortalité est inférieur à la moyenne française (2,74 %), avec une part des victimes qui a été tuées de 1,36 %. Les accidents se produisent principalement aux alentours de 18 heures. Parmi les personnes impliquées dans un accident nocturne, 1,81 % ont perdu la vie, ce qui est 1,5 fois plus élevé que le taux des accidents matinaux (1,17 %). Enfin, parmi ceux impliqués hors agglomération, 3,17 % ont été tués, soit 3,9 fois plus que dans les zones urbaines.
✅ Tous les nombres du texte viennent de la fiche.

=============== qwen2.5:3b · département 75
Le département 75 a enregistré 4 191 accidents en 2024, impliquant 9 132 personnes dont 31 ont été tuées, ce qui représente une part de 0,34 % des victimes, inférieure à la moyenne française (2,74 %). Les accidents se produisent principalement aux alentours de 18 heures. Parmi les p